---
title: Webscraping
author: "Deepika Agarwal"
format:
  html:
    embed-resources: true
echo: true
---


# XML, HTML, and Web Scraping

JSON and XML are two different ways to represent hierarchical data. Which one is better? There are lots of articles online which discuss similarities and differences between JSON and XML and their advantages and disadvantages. Both formats are still in current usage, so it is good to be familiar with both. However, JSON is more common, so we'll focus on working with JSON representations of hierarchical data.

The reading covered an example of using Beautiful Soup to parse XML. Rather than doing another example XML now, we'll skip straight to scraping HTML from a webpage. Both HTML and XML can be parsed in a similar way with Beautiful Soup.

In [114]:
import pandas as pd
import requests

## Scraping an HTML table with Beautiful Soup

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2022). We'll use Beautiful Soup to scrape information from this table.

Read in the HTML from the URL using the `requests` library.

In [115]:
# YOUR CODE HERE
URL = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(URL, headers = HEADERS)

In [116]:
response

<Response [200]>

Use Beautiful Soup to parse this string into a tree called `soup`

In [117]:
# YOUR CODE HERE
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.content, "html.parser")

To find an HTML tag corresponding to a specific element on a webpage, right-click on it and choose "Inspect element". Go to the cities table Wikipedia page and do this now.

You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="wikitable sortable jquery-tablesorter" style="text-align:center">
```

There are many `<table>` tags on the page.

In [118]:
len(soup.find_all("table"))

10

Identified element of the cities table:

```html
<table class="wikitable sortable sort-under col1left col2center jquery-tablesorter" style="text-align:right">
```

We can use attributes like `class=` and `style=` to narrow down the list.

In [119]:
len(soup.find_all("table",
                  attrs={
                      "class": "wikitable sortable sort-under col1left col2center",
                      "style": "text-align:right"}
                  ))

1

At this point, you can manually inspect the tables on the webpage to find that the one we want is the first one (see `[0]` below). We'll store this as `table`.

In [120]:
table = soup.find_all("table",
                  attrs={
                      "class": "wikitable sortable sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

In [121]:
table

<table class="wikitable sortable sort-under col1left col2center" style="text-align:right">
<tbody><tr>
<th>City
</th>
<th><abbr title="State or district">ST</abbr>
</th>
<th>2024<br/>estimate
</th>
<th>Peak<br/>population
</th>
<th>%<br/>decline<br/>from peak
</th>
<th>Peak<br/>year
</th>
<th class="unsortable">
</th></tr>
<tr>
<td><a href="/wiki/Allegheny,_Pennsylvania" title="Allegheny, Pennsylvania">Allegheny</a></td>
<td><a href="/wiki/Pennsylvania" title="Pennsylvania">PA</a></td>
<td>NA</td>
<td>129,896</td>
<td>NA</td>
<td>1907</td>
<td><sup class="reference" id="cite_ref-33"><a href="#cite_note-33"><span class="cite-bracket">[</span>ab<span class="cite-bracket">]</span></a></sup>
</td></tr>
<tr>
<td><a href="/wiki/Brooklyn" title="Brooklyn">Brooklyn</a></td>
<td><a href="/wiki/New_York_(state)" title="New York (state)">NY</a></td>
<td>NA</td>
<td>806,343</td>
<td>NA</td>
<td>1898</td>
<td><sup class="reference" id="cite_ref-34"><a href="#cite_note-34"><span class="cite-bracket"

**Now you will write code to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for: city, state, population (2022 estimate), and 2020 land area (sq mi).** Refer to the Notes/suggestions below as you write your code. A few Hints are provided further down, but try coding first before looking at the hints.

Notes/suggestions:

- Use as a guide the code from the reading that produced the data frame of Statistics faculty
- Inspect the page source as you write your code
- You will need to write a loop to get the information for all cities, but you might want to try just scraping the info for New York first
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `.get_text(strip = True)` instead of `.text`
- Don't forget to convert to a Pandas Data Frame; it should have 333 rows and 4 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it --- e.g., what is the population density for all cities in CA? --- then you would need to clean the data first (to clean strings and convert to quantitative). (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)

In [122]:
# YOUR CODE HERE. ADD AS MANY CELLS AS NEEDED
# initialize an empty list
rows = []

# iterate over all rows in the table
for city in table.find_all("tr")[1:]:

    # Get all the cells (<td>) in the row.
    cells = city.find_all("td")

    # The information we need is the text between tags.

    # Find the city name in cell[0]
    name_tag = cells[0]
    name = name_tag.get_text(strip = True)

    # Find the office of the faculty in cell[1]
    # which for most faculty is contained in the <a> tag
    state_tag = cells[1].find("a") or cells[1]
    state = state_tag.get_text(strip = True)

    # Find the email of the faculty in cell[3]
    # which for most faculty is contained in the <a> tag
    pop_tag = cells[3].find("a") or cells[2]
    pop = pop_tag.get_text(strip = True)

    # Append this data.
    rows.append({
        "City": name,
        "State": state,
        "2024 Population": pop
    })

In [123]:
df_cities = pd.DataFrame(rows)
df_cities

,City,State,2024 Population
0,Allegheny,PA,NA
1,Brooklyn,NY,NA
2,Camden,NJ,"71,749"
3,Canton,OH,"69,211"
4,Citrus Heights,CA,"86,909"
5,Duluth,MN,"87,986"
6,Erie,PA,"92,940"
7,Fall River,MA,"94,689"
8,Flint,MI,"79,735"
9,Gary,IN,"67,555"


Hints:

- Each city is a row in the table; find all the `<tr>` tags to find all the cities
- Look for the `<td>` tag to see table entries within a row
- The rank column is represented by `<th>` tags, rather than `<td>` tags. So within a row, the first (that is, `[0]`) `<td>` tag corresponds to the city name.

## Aside: Scraping an HTML table with Pandas



The Pandas command `read_html` can be used to scrape information from an HTML table on a webpage.

We can call `read_html` on the URL.

In [124]:
pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population")

HTTPError: HTTP Error 403: Forbidden

However, this scrapes all the tables on the webpage, not just the one we want. As with Beautiful Soup, we can narrow the search by specifying the table attributes.

In [ ]:
pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", attrs = {'class': 'wikitable sortable', "style": "text-align:center"})

This still returns 3 tables. As we remarked above, the table that we want is the first one (see `[0]` below).

In [ ]:
df_cities2 = pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", attrs = {'class': 'wikitable sortable', "style": "text-align:center"})[0]
df_cities2

Wait, that seemed much easier than using Beautiful Soup, and it returned a data frame, and we even got for free some formatting like removing the commas from the population! Why didn't we just use `read_html` in the first place? It's true the `read_html` works well when scraping information from an HTML *table*. Unfortunately, you often want to scrape information from a webpage that isn't conveniently stored in an HTML table, in which case `read_html` won't work. (It only searches for `<table>`, `<th>`, `<tr>`, and `<td>` tags, but there are many other HTML tags.) Though Beautiful Soup is not as simple as `read_html`, it is more flexible and thus more widely applicable.

## Scraping information that is NOT in a `<table>` with Beautiful Soup

The Cal Poly course catalog http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory contains a list of courses offered by the Statistics department. **You will scrape this website to obtain a Pandas data frame with one row for each DATA or STAT course and two columns: course name and number (e.g, DATA 301. Introduction to Data Science) and term typically offered (e.g., Term Typically Offered: F, W, SP).**

Note: Pandas `read_html` is not help here since the courses are not stored in a `<table>.`

In [ ]:
pd.read_html("http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory")


Notes/suggestions:


- Inspect the page source as you write your code
- The courses are not stored in a `<table>`. How are they stored?
- You will need to write a loop to get the information for all courses, but you might want to try just scraping the info for DATA 100 first
- What kind of tag is the course name stored in? What is the `class` of the tag?
- What kind of tag is the quarter(s) the course is offered stored in? What is the `class` of the tag? Is this the only tag of this type with the class? How will you get the one you want?
- You don't have to remove the number of units (e.g., 4 units) from the course name and number, but you can try it if you want
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `get_text(strip = True)` instead of `text`
- Don't forget to convert to a Pandas Data Frame; it should have 74 rows and 2 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it then you might need to clean the data first. (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)



In [ ]:
# YOUR CODE HERE
URL = "https://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory"

response = requests.get(URL)
response

In [ ]:
soup = BeautifulSoup(response.content, "html.parser")

In [ ]:
# initialize an empty list to store all course data
rows = []

# loop through every course block on the page
for courses in soup.find_all("div", {"class": "courseblock"}):

    # find the <p> tag that contains the course title
    names = courses.find_all("p", {"class": "courseblocktitle"})

    # remove the <span> tag inside it that contains the units
    for span in names[0].find_all("span", {"class": "courseblockhours"}):
        span.extract()

    # get the cleaned course title text (no "4 units", no HTML tags)
    names_clean = names[0].get_text(strip=True)

    # find the <p> tag that contains when the course is offered
    term = courses.find_all("p", {"class": "noindent"})[0].get_text(strip=True)

    # append this course's information
    rows.append({
        "Course # and Name": names_clean,
        "When Offered": term
    })

df_courses = pd.DataFrame(rows)
df_courses


Hints:

- Each course is represented by a `<div>` with `class=courseblock`, so you can find all the courses with `soup.find_all("div", {"class": "courseblock"})`
- The course name is in a `<p>` tag with `class=courseblocktitle`, inside a `<strong>` tag. (Though I don't think we need to find the strong tag here.)
- The term typically offered is in `<p>` tag with `class=noindent`. However, there are several tags with this class; term typically offered is the first one.
- If you want to use Beautiful Soup to remove the course units (e.g., 4 units), find the `<span>` tag within the course name tag and `.extract()` this span tag